In [ ]:
import os
from astropy.convolution import convolve_fft, Gaussian2DKernel

import scipy.ndimage as nd

import matplotlib.pyplot as plt
import matplotlib.colors as colors
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.ticker as ticker
from matplotlib import colormaps
from matplotlib.colors import ListedColormap
import matplotlib.cm as cm


import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.time import Time
from astropy.visualization import ImageNormalize, SqrtStretch
from sunpy.coordinates.ephemeris import get_body_heliographic_stonyhurst
from astropy.coordinates import solar_system_ephemeris
from astropy.modeling.functional_models import Disk2D

import sunpy.coordinates  as coord # NOQA
import sunpy.map
from sunpy.net import Fido
from sunpy.net import attrs as a
from sunpy.coordinates import frames

from sunkit_image.coalignment import mapsequence_coalign_by_match_template as mc_coalign
from sunkit_image.coalignment import calculate_match_template_shift as mc_shift

import numpy as np

import cmasher as cmr

from identification_utils import *

from datetime import datetime, timedelta
from astropy.io import fits

/home/sophie-stucki/anaconda3/envs/sunpy/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
###Parameters to load data (see sunpy tutorial)

#mail access
jsoc_email = "stucki@ieec.cat"


#data names
Ic_serie ='hmi.Ic_noLimbDark_720s'
M_serie = 'hmi.M_45s'

#directory to retrieve the files into
path = '/home/sophie-stucki/starsim/starsim/SDO_input/SDO_images'
maps_path = '/home/sophie-stucki/starsim/starsim/SDO_input/maps/'

#False if the data are already download
download_from_sdo = False

In [6]:
# Set the date range you want to loop over
start_date = datetime.strptime('2016-01-01', '%Y-%m-%d')
end_date = datetime.strptime('2017-09-01', '%Y-%m-%d')  # loop will include up to 2017-08-27

cont_map_list = []
los_map_list = []
t_list = []


# Loop day by day
current_date = start_date
while current_date < end_date:
    # Time range for this day
    start_time = current_date.replace(hour=0, minute=0, second=0)
    end_time = start_time + timedelta(seconds=30)

    # Format if needed
    start_str = start_time.strftime('%Y-%m-%dT%H:%M:%S')
    end_str = end_time.strftime('%Y-%m-%dT%H:%M:%S')

    print(f"Processing {start_str} to {end_str}")

    # Put your SunPy/Fido data access or map logic here

    ###Load the data
    date_str = current_date.strftime('%Y%m%d')

    if download_from_sdo:
        cont_sequence, los_sequence = load_data(start_time, end_time, jsoc_email, Ic_serie, M_serie, path=path)

        cont_map = cont_sequence.maps[0]
        los_map = los_sequence.maps[0]

    else:      
        
        cont_filename = f"{path}/hmi.ic_nolimbdark_720s.{date_str}_000000_TAI.3.continuum.fits"
        los_filename = f"{path}/hmi.m_45s.{date_str}_000045_TAI.2.magnetogram.fits"

        if os.path.exists(cont_filename) and os.path.exists(los_filename):
            cont_fits = fits.open(cont_filename)
            cont_map = sunpy.map.Map(cont_fits[1].data,cont_fits[1].header, sequence=False, allow_errors=True)

            los_fits = fits.open(los_filename)
            los_map = sunpy.map.Map(los_fits[1].data, los_fits[1].header, sequence=True, allow_errors=True)
        

    t_list.append(date_str)
    cont_map_list.append(cont_map)
    los_map_list.append(los_map)
       
    
    # two_graphs_plot(cont_map, los_map)

    # Move to the next day
    current_date += timedelta(days=1)

Processing 2016-01-01T00:00:00 to 2016-01-01T00:00:30


NameError: name 'cont_map' is not defined

In [ ]:
# Downsampling
out_shape = (1024, 1024)

# Noise threshold (from Sen & al. 2023)
noise_thresh = 8

In [ ]:
for i in range(len(t_list)):

    t = t_list[i]
    cont_map = cont_map_list[i]
    los_map = los_map_list[i][0]

    print(f"Processing {t}")

    RAW_size = cont_map.dimensions

    # downsampling
    cont_map = cont_map.resample(out_shape * u.pix)
    los_map = los_map.resample(out_shape * u.pix)   


    # coordinates
    xg, yg = coord_grid(los_map)

    # remove the noise
    los_sequence_updated = sunpy.map.MapSequence(noise_threshold(los_map, threshold=noise_thresh)) 
    # remove the foreshortening effects
    los_sequence_updated = sunpy.map.MapSequence(removing_foreshortening_effect(los_sequence_updated.maps[0],xg,yg))

    # crop
    min_p = int(np.argwhere(cont_map.data[int(out_shape[0]/2), :] >= 0 ).min() - 1)
    max_p = int(np.argwhere(cont_map.data[int(out_shape[0]/2), :] >= 0 ).max() + 1)

    cont_map = sunpy.map.sources.HMIMap(cont_map.data[min_p:max_p, min_p:max_p], cont_map.fits_header)
    los_map = sunpy.map.sources.HMIMap(los_sequence_updated.maps[0].data[min_p:max_p, min_p:max_p], los_sequence_updated.maps[0].fits_header)

    xg = xg[min_p:max_p, min_p:max_p]
    yg = yg[min_p:max_p, min_p:max_p]

    # plot the new sdo images
    # two_graphs_plot(cont_map, los_map)

    ### identification following Sen & al. 2023

    active_area = active_area_identification(cont_map, los_map, xg, yg)
    spot_area, smooth_spot_area, feature_area, smooth_feature_area = active_area_smoothing(active_area, spot_threshold=0.2, spot_kernel_size=2, feature_threshold=0.2, feature_kernel_size=2) 

    area_th = micro_hemisphere_to_arcsec2(los_sequence_updated.maps[0], 20).value
    identification, plage_nbr = network_identification(feature_area, smooth_feature_area, los_map.scale[0].value, area_th, method='scipy')
    nbr, locs, pxl_area = spot_nbr(spot_area, smooth_spot_area)

    # plot the identification map
    # identifiation_plot(identification, spot_area, cont_map, locs, filename=path+'identification_map_downsampling_{}_{}.pdf'.format(int(RAW_size[0].value/out_shape[0]), t))

    ### save the faculae and spot maps for starsim

    facula_map = np.copy(identification)

    facula_map[np.isnan(facula_map)] = 0

    # facula_map = np.zeros(np.shape(identification))
    facula_map[np.isnan(cont_map.data)] = np.nan

    # spot_map = np.copy(spot_area)
    spot_map = np.zeros(np.shape(spot_area))
    
    spot_map[np.isnan(spot_map)] = 0
    spot_map[np.isnan(cont_map.data)] = np.nan

    facula_map = np.flip(facula_map)
    spot_map = np.flip(spot_map)


    np.savetxt(maps_path+'faculae_map_{:.1f}.txt'.format(i), facula_map)
    np.savetxt(maps_path+'spot_map_{:.1f}.txt'.format(i), spot_map)
    np.savetxt(maps_path+'cont_map_{:.1f}.txt'.format(i), np.flip(cont_map.data))
    np.savetxt(maps_path+'los_map_{:.1f}.txt'.format(i), np.flip(los_map.data))



    



NotImplementedError: The ability to index Map by physical coordinate is not yet implemented.